In [0]:
CREATE OR REPLACE VIEW workspace.default.zip_location_type_summary_v2 AS
WITH location_counts AS (
  SELECT
    zip_code,
    COALESCE(location_type, 'Unknown') AS location_type,
    COUNT(*) AS resident_rat_report_count
  FROM workspace.default.rat_clean_v2
  WHERE resident_rat_report
  GROUP BY
    zip_code,
    COALESCE(location_type, 'Unknown')
)

SELECT
  zip_code,
  location_type,
  resident_rat_report_count,

  SUM(resident_rat_report_count) OVER (
    PARTITION BY zip_code
  ) AS zip_resident_rat_report_count,

  ROUND(
    100.0 * resident_rat_report_count
      / SUM(resident_rat_report_count) OVER (
          PARTITION BY zip_code
        ),
    2
  ) AS location_share_percent

FROM location_counts;

CREATE OR REPLACE VIEW workspace.default.zip_descriptor_summary_v2 AS
WITH descriptor_counts AS (
  SELECT
    zip_code,
    descriptor,
    provenance_type,
    COUNT(*) AS record_count
  FROM workspace.default.rat_clean_v2
  GROUP BY
    zip_code,
    descriptor,
    provenance_type
)

SELECT
  zip_code,
  descriptor,
  provenance_type,
  record_count,

  SUM(record_count) OVER (
    PARTITION BY zip_code
  ) AS zip_total_record_count,

  ROUND(
    100.0 * record_count
      / SUM(record_count) OVER (
          PARTITION BY zip_code
        ),
    2
  ) AS descriptor_share_percent

FROM descriptor_counts;

SELECT *
FROM workspace.default.zip_descriptor_summary_v2
WHERE zip_code = '10035'
ORDER BY record_count DESC;